# Respiratory Rate Extraction from PPG using RRest

**Purpose**: Extract respiratory rate features from raw PPG (ppg2_green_6) using the RRest toolbox.

**Method**: 
- Uses RRest v3.0 with optimized configuration (AM, FM, BW features)
- Processes 120-second windows with 60-second RRest sub-windows
- Extracts 5 respiratory features per window: mean, std, max, min, trend

**Output**: `reports/rr_features_from_ppg.csv`

**Author**: Generated from best practices discussion

**Date**: January 1, 2026


In [14]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Matlab engine
import matlab.engine

# Add shared to path
sys.path.insert(0, str(Path.cwd().parent / "experiments" / "shared"))

from raw_loader import load_raw_signals, get_all_subjects, get_experiment_time_range
from windowing import parse_stress_events
from config import DEFAULT_CONFIG

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ Imports successful")


✓ Imports successful


In [15]:
# Paths
RREST_PATH = Path.cwd().parent / "experiments" / "shared" / "RRest" / "RRest" / "RRest_v3.0"
CONFIG_PATH = Path.cwd().parent / "backup_code" / "matlabsettings.md"
OUTPUT_PATH = Path.cwd().parent / "reports" / "rr_features_from_ppg.csv"

# Processing parameters
config = DEFAULT_CONFIG
WINDOW_SIZE_SEC = 120  # 120-second stress prediction windows
RREST_WINDOW_SEC = 60  # 60-second RRest sub-windows (recommended)
PPG_SAMPLING_RATE = 64.0  # Hz
MIN_PPG_COVERAGE = 0.5  # Require 50% PPG data in window

print(f"RRest path: {RREST_PATH}")
print(f"RRest exists: {RREST_PATH.exists()}")
print(f"Output path: {OUTPUT_PATH}")
print(f"\nProcessing parameters:")
print(f"  Window size: {WINDOW_SIZE_SEC}s")
print(f"  RRest sub-window: {RREST_WINDOW_SEC}s")
print(f"  PPG sampling rate: {PPG_SAMPLING_RATE} Hz")


RRest path: /Users/jithuazeez/Documents/Msc/Dissertation/experiments/shared/RRest/RRest/RRest_v3.0
RRest exists: True
Output path: /Users/jithuazeez/Documents/Msc/Dissertation/reports/rr_features_from_ppg.csv

Processing parameters:
  Window size: 120s
  RRest sub-window: 60s
  PPG sampling rate: 64.0 Hz


In [16]:
class RRestMatlabWrapper:
    """Wrapper to call RRest v3.0 via Matlab Engine."""
    
    def __init__(self, rrest_path: Path, window_length: int = 60):
        """
        Initialize RRest wrapper.
        
        Args:
            rrest_path: Path to RRest v3.0 directory
            window_length: RRest window length in seconds (default: 60)
        """
        self.rrest_path = rrest_path
        self.window_length = window_length
        self.eng = None
        self.temp_data_path = Path("/tmp/rrest_temp_data")
        self.temp_data_path.mkdir(exist_ok=True)
        
        print("Initializing Matlab engine...")
        try:
            self.eng = matlab.engine.start_matlab()
            # Suppress MATLAB output
            self.eng.eval('warning off all', nargout=0)
            print("✓ Matlab engine started")
            
            # Add RRest to path
            self.eng.addpath(str(self.rrest_path), nargout=0)
            self.eng.addpath(str(self.rrest_path / 'Algorithms'), nargout=0)
            print(f"✓ Added RRest to Matlab path: {self.rrest_path}")
            
        except Exception as e:
            print(f"❌ Failed to start Matlab engine: {e}")
            raise
    
    def setup_rrest_config(self):
        """
        Configure RRest with optimized settings for PPG.
        RRest v3.0 reads configuration from setup_universal_params.m
        We need to modify that file or create our own.
        """
        print("\nConfiguring RRest...")
        
        # RRest v3.0 uses setup_universal_params.m
        # We'll create a custom setup function in Matlab workspace
        config_commands = f"""
        % Create setup_universal_params function in workspace
        % This will be called by RRest
        
        function up = setup_universal_params
            fprintf('Loading custom universal parameters...\\n');
            
            % Paths - point to temp directory
            up.paths.root_folder = '{str(self.temp_data_path)}';
            up.paths.data_save_folder = '{str(self.temp_data_path)}';
            up.paths.analysis_path = '{str(self.temp_data_path)}';
            
            % Algorithm stages
            up.al.key_components = {{'extract_resp_sig', 'estimate_rr', 'fuse_rr'}};
            
            % Stage 1: Extraction (feature-based for PPG)
            up.al.options.extract_resp_sig = {{'ppg_feat'}};
            up.al.sub_components.ppg_feat = {{'EHF', 'PDt', 'FPt', 'FMe', 'RS', 'ELF'}};
            
            % Beat detection (IMS for PPG)
            up.al.options.PDt = {{'IMS'}};
            
            % Fiducial points (all available)
            up.al.options.FPt = {{'all'}};
            
            % Feature measurement (AM, FM, BW - fast subset)
            up.al.options.FMe = {{'am', 'fm', 'bw'}};
            
            % Resampling (cubic with bandpass)
            up.al.options.RS = {{'cubB'}};
            
            % Stage 2: RR Estimation (time-domain only - faster)
            up.al.options.estimate_rr = {{'CtO', 'CtA', 'PKS', 'ZeX', 'PZX'}};
            
            % Stage 3: Fusion
            up.al.options.fuse_rr = {{'fus_mod'}};
            up.al.sub_components.fus_mod = {{'SFu'}};
            
            % Window parameters
            up.paramSet.winLeng = {self.window_length};
            up.paramSet.winOverlap = 0;
            up.paramSet.rr_range = [4, 60];
            
            % Analysis options
            up.analysis.prelim = 0;  % Don't do preliminary analysis
            up.analysis.stats = 0;   % Don't do statistical analysis
            up.analysis.level = 'multiple_breaths';  % Analyze multiple breaths
            
            fprintf('✓ Custom parameters loaded\\n');
        end
        
        % Test the function
        test_up = setup_universal_params;
        disp('✓ RRest configuration ready');
        """
        
        # Execute configuration
        self.eng.eval(config_commands, nargout=0)
        print("✓ RRest configured with optimized settings")
        print(f"  - Features: AM, FM, BW")
        print(f"  - Estimation: Time-domain (5 methods)")
        print(f"  - Fusion: Smart Fusion")
        print(f"  - Window: {self.window_length}s")
    
    def extract_rr_from_window(self, ppg_values: np.ndarray, fs: float = 64.0) -> dict:
        """
        Extract respiratory rate from PPG window using RRest v3.0.
        
        Args:
            ppg_values: 1D array of PPG values
            fs: Sampling rate in Hz
        
        Returns:
            Dictionary with RR estimates or None if failed
        """
        if len(ppg_values) < fs * 30:  # Need at least 30 seconds
            return None
        
        try:
            # RRest v3.0 expects data saved as .mat file with specific structure
            # Create data structure and save it
            dataset_name = "temp_ppg_data"
            mat_file = self.temp_data_path / f"{dataset_name}_data.mat"
            
            # Save PPG data in format expected by RRest
            self.eng.workspace['ppg_values'] = matlab.double(ppg_values.tolist())
            self.eng.workspace['fs'] = int(fs)
            self.eng.workspace['mat_file'] = str(mat_file)
            
            # Create and save data structure
            self.eng.eval("""
            % Create data structure for RRest
            data = struct();
            data(1).ppg.v = ppg_values(:)';  % Row vector
            data(1).ppg.fs = int32(fs);
            data(1).group = 'ppg';  % Required field
            
            % Save to mat file
            save(mat_file, 'data');
            """, nargout=0)
            
            # Run RRest on the saved data
            # RRest(dataset_name) ONLY - it calls setup_universal_params internally
            try:
                self.eng.eval(f"RRest('{dataset_name}');", nargout=0, timeout=30.0)
                
                # Load the results file that RRest saves
                results_file = self.temp_data_path / f"{dataset_name}_win_data.mat"
                
                if results_file.exists():
                    # Load the results
                    results = self.eng.load(str(results_file), nargout=1)
                    
                    # Extract RR estimates from the win_data structure
                    # win_data contains: win_data.rrEst (RR estimates per window)
                    rr_estimates = np.array(self.eng.eval(
                        f"load('{str(results_file)}'); win_data.rrEst(:,1)", nargout=1
                    )).flatten()
                    
                    # Clean up results file
                    results_file.unlink()
                    
                    if len(rr_estimates) > 0:
                        # Filter physiologically valid estimates (4-60 bpm)
                        valid_rr = rr_estimates[(rr_estimates >= 4) & (rr_estimates <= 60) & (~np.isnan(rr_estimates))]
                        print()
                        
                        if len(valid_rr) > 0:
                            return {
                                'rr_mean': float(np.mean(valid_rr)),
                                'rr_std': float(np.std(valid_rr)) if len(valid_rr) > 1 else 0.0,
                                'rr_min': float(np.min(valid_rr)),
                                'rr_max': float(np.max(valid_rr)),
                                'rr_trend': float(valid_rr[-1] - valid_rr[0]) if len(valid_rr) > 1 else 0.0,
                                'n_estimates': len(valid_rr),
                                'quality': min(1.0, len(valid_rr) / 3.0)
                            }
            except Exception as e:
                # RRest might fail silently
                return None
            finally:
                # Clean up temp files
                if mat_file.exists():
                    mat_file.unlink()
                # Clean up any other temp files RRest might create
                for f in self.temp_data_path.glob(f"{dataset_name}*"):
                    try:
                        f.unlink()
                    except:
                        pass
            
            return None
            
        except Exception as e:
            print(f"RRest extraction error: {e}")
            return None
    
    def close(self):
        """Close Matlab engine."""
        if self.eng:
            print("\nClosing Matlab engine...")
            self.eng.quit()
            print("✓ Matlab engine closed")

print("✓ RRestMatlabWrapper class defined")


✓ RRestMatlabWrapper class defined


In [17]:
def extract_rr_for_subject(subject_folder: Path, 
                          rrest_wrapper,
                          config) -> pd.DataFrame:
    """
    Extract RR features for all 120s windows in a subject.
    
    Uses 60s sub-windows for RRest, then aggregates to 120s windows.
    """
    signals = load_raw_signals(subject_folder)
    subject_id = signals["subject_id"]
    
    # Check PPG availability
    ppg_df = signals.get("ppg")
    if ppg_df is None or len(ppg_df) == 0:
        return pd.DataFrame()
    
    # Get time range
    try:
        start, end = get_experiment_time_range(signals)
    except ValueError:
        return pd.DataFrame()
    
    # Create 120s windows
    duration_sec = (end - start).total_seconds()
    window_size_sec = WINDOW_SIZE_SEC
    overlap_ratio = config.overlap_ratio
    overlap_sec = int(window_size_sec * overlap_ratio)
    step_sec = window_size_sec - overlap_sec
    
    # Generate window starts
    skip_sec = config.skip_first_minutes * 60
    window_starts = []
    current_time = start + pd.Timedelta(seconds=skip_sec)
    
    while current_time + pd.Timedelta(seconds=window_size_sec) <= end:
        window_starts.append(current_time)
        current_time += pd.Timedelta(seconds=step_sec)
    
    # Extract RR for each window
    rr_rows = []
    
    for window_start in window_starts:
        window_end = window_start + pd.Timedelta(seconds=window_size_sec)
        
        # Extract PPG for this 120s window
        ppg_mask = (ppg_df["timestamp"] >= window_start) & (ppg_df["timestamp"] < window_end)
        ppg_window = ppg_df[ppg_mask]
        
        # Check coverage
        expected_samples = window_size_sec * PPG_SAMPLING_RATE
        if len(ppg_window) < MIN_PPG_COVERAGE * expected_samples:
            continue
        
        # Get PPG values and remove NaN/zeros
        ppg_values = ppg_window["value"].values
        ppg_values = ppg_values[(~np.isnan(ppg_values)) & (ppg_values != 0)]
        
        if len(ppg_values) < MIN_PPG_COVERAGE * expected_samples:
            continue
        
        # Split into two 60s sub-windows for RRest
        mid_point = len(ppg_values) // 2
        sub_window_1 = ppg_values[:mid_point]
        sub_window_2 = ppg_values[mid_point:]
        
        # Extract RR from each sub-window
        rr_1 = rrest_wrapper.extract_rr_from_window(sub_window_1, fs=PPG_SAMPLING_RATE)
        rr_2 = rrest_wrapper.extract_rr_from_window(sub_window_2, fs=PPG_SAMPLING_RATE)
        
        # Aggregate if we have at least one estimate
        valid_estimates = []
        if rr_1: valid_estimates.append(rr_1['rr_mean'])
        if rr_2: valid_estimates.append(rr_2['rr_mean'])
        
        if len(valid_estimates) > 0:
            row = {
                'subject_id': subject_id,
                'window_start': window_start,
                'window_end': window_end,
                'rr_mean': np.mean(valid_estimates),
                'rr_std': np.std(valid_estimates) if len(valid_estimates) > 1 else 0.0,
                'rr_min': np.min(valid_estimates),
                'rr_max': np.max(valid_estimates),
                'rr_trend': valid_estimates[-1] - valid_estimates[0] if len(valid_estimates) > 1 else 0.0,
                'n_sub_windows': len(valid_estimates),
                'quality': len(valid_estimates) / 2.0,  # Both sub-windows = quality 1.0
                'ppg_coverage': len(ppg_values) / expected_samples
            }
            rr_rows.append(row)
    
    return pd.DataFrame(rr_rows)

print("✓ Helper functions defined")


✓ Helper functions defined


In [18]:
# Initialize RRest
print("="*70)
print("RESPIRATORY RATE EXTRACTION FROM PPG USING RREST v3.0")
print("="*70)

rrest = RRestMatlabWrapper(RREST_PATH, window_length=RREST_WINDOW_SEC)
# rrest.setup_rrest_config()


RESPIRATORY RATE EXTRACTION FROM PPG USING RREST v3.0
Initializing Matlab engine...
✓ Matlab engine started
✓ Added RRest to Matlab path: /Users/jithuazeez/Documents/Msc/Dissertation/experiments/shared/RRest/RRest/RRest_v3.0


In [19]:
# Get all subjects
subjects = get_all_subjects(config.data_path)
print(f"\nFound {len(subjects)} subjects")
print(f"Processing with {WINDOW_SIZE_SEC}s windows, {RREST_WINDOW_SEC}s RRest sub-windows")
print()



Found 21 subjects
Processing with 120s windows, 60s RRest sub-windows



In [20]:
# Extract RR for all subjects
all_rr_data = []
failed_subjects = []

for subject_folder in tqdm(subjects, desc="Processing subjects"):
    try:
        rr_df = extract_rr_for_subject(subject_folder, rrest, config)
        
        if len(rr_df) > 0:
            all_rr_data.append(rr_df)
        else:
            failed_subjects.append(subject_folder.name)
    except Exception as e:
        print(f"\nError processing {subject_folder.name}: {e}")
        failed_subjects.append(subject_folder.name)
        # break

# Close Matlab engine
rrest.close()

print(f"\n{'='*70}")
print(f"EXTRACTION COMPLETE")
print(f"{'='*70}")
print(f"Successful: {len(all_rr_data)} subjects")
print(f"Failed: {len(failed_subjects)} subjects")
if failed_subjects:
    print(f"Failed subjects: {', '.join([s[:12] for s in failed_subjects[:5]])}...")


Processing subjects:   0%|          | 0/21 [00:00<?, ?it/s]


Closing Matlab engine...
✓ Matlab engine closed

EXTRACTION COMPLETE
Successful: 0 subjects
Failed: 21 subjects
Failed subjects: id_0a73ef1b-, id_3e775b57-, id_3f27501c-, id_3f62db18-, id_464cc459-...


In [9]:
# Combine all subjects
if all_rr_data:
    rr_df = pd.concat(all_rr_data, ignore_index=True)
    
    # Save to CSV
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    rr_df.to_csv(OUTPUT_PATH, index=False)
    
    print(f"✓ Saved to: {OUTPUT_PATH}")
    print(f"  Total windows: {len(rr_df)}")
    print(f"  Subjects: {rr_df['subject_id'].nunique()}")
    print(f"  File size: {OUTPUT_PATH.stat().st_size / 1024:.1f} KB")
else:
    print("❌ No data extracted!")
    rr_df = None


❌ No data extracted!


---
## 🔧 EXPORT DATA FOR MANUAL MATLAB TESTING

Export PPG data in the exact format RRest expects, so you can test processing it manually in MATLAB.


In [12]:
# Export PPG data in MATLAB format for manual testing
import scipy.io as sio

print("="*70)
print("EXPORTING PPG DATA FOR MATLAB/RREST")
print("="*70)

# Get first subject
subjects = get_all_subjects(config.data_path)
test_subject = subjects[0]
print(f"\nUsing subject: {test_subject.name}")

# Load signals
signals = load_raw_signals(test_subject)
ppg_df = signals["ppg"]
start, end = get_experiment_time_range(signals)

# Extract multiple 120s windows
n_windows = 3  # Export 3 windows for testing
window_size_sec = 120
data_list = []

print(f"\nExtracting {n_windows} windows of {window_size_sec}s each...")

for i in range(n_windows):
    window_start = start + pd.Timedelta(seconds=60 + i*120)  # Start after 1 min
    window_end = window_start + pd.Timedelta(seconds=window_size_sec)
    
    # Extract PPG
    ppg_mask = (ppg_df["timestamp"] >= window_start) & (ppg_df["timestamp"] < window_end)
    ppg_window = ppg_df[ppg_mask]
    
    if len(ppg_window) > 0:
        ppg_values = ppg_window["value"].values
        # Remove NaN and zeros
        ppg_values = ppg_values[(~np.isnan(ppg_values)) & (ppg_values != 0)]
        
        if len(ppg_values) >= 1920:  # At least 30s of data
            print(f"  Window {i+1}: {len(ppg_values)} samples ({len(ppg_values)/64:.1f}s)")
            data_list.append({
                'ppg_v': ppg_values,
                'ppg_fs': 64,
                'group': 'test_ppg'
            })

print(f"\n✓ Extracted {len(data_list)} valid windows")

# Create MATLAB structure in the format RRest expects
# Format: data(n).ppg.v, data(n).ppg.fs, data(n).group
matlab_data = {}

for i, window_data in enumerate(data_list, 1):
    # MATLAB uses 1-based indexing, but scipy.io handles the conversion
    # We create a structured array
    ppg_struct = {
        'v': window_data['ppg_v'].reshape(1, -1),  # Row vector
        'fs': np.array([[window_data['ppg_fs']]], dtype=np.int32)
    }
    
    matlab_data[f'data_{i}'] = {
        'ppg': ppg_struct,
        'group': window_data['group']
    }

# Save to .mat file
output_mat_path = Path.cwd().parent / "reports" / "test_ppg_for_rrest.mat"
output_mat_path.parent.mkdir(parents=True, exist_ok=True)

# Convert to proper MATLAB structure format
# We need to create a structure array
data_array = []
for i, window_data in enumerate(data_list):
    data_array.append({
        'ppg': np.array([(window_data['ppg_v'].reshape(1, -1), 
                         np.array([[window_data['ppg_fs']]], dtype=np.int32))],
                       dtype=[('v', 'O'), ('fs', 'O')]),
        'group': window_data['group']
    })

# Create the structure array the way MATLAB expects
# data(1).ppg.v, data(1).ppg.fs, data(1).group
matlab_struct = np.zeros((1, len(data_list)), dtype=np.object_)
for i, window_data in enumerate(data_list):
    ppg_dtype = np.dtype([('v', 'O'), ('fs', 'O')])
    ppg_struct = np.array([(window_data['ppg_v'].reshape(1, -1), 
                           np.int32(window_data['ppg_fs']))], 
                         dtype=ppg_dtype)
    
    data_dtype = np.dtype([('ppg', 'O'), ('group', 'O')])
    matlab_struct[0, i] = np.array([(ppg_struct, window_data['group'])], 
                                   dtype=data_dtype)

# Save with scipy.io
sio.savemat(output_mat_path, {'data': matlab_struct}, oned_as='row')

print(f"\n✓ Saved MATLAB file: {output_mat_path}")
print(f"  File size: {output_mat_path.stat().st_size / 1024:.1f} KB")
print(f"  Number of recordings: {len(data_list)}")

print("\n" + "="*70)
print("HOW TO USE IN MATLAB")
print("="*70)
print(f"""
1. Open MATLAB

2. Load the data:
   >> load('{output_mat_path}')
   >> whos data

3. Inspect the structure:
   >> data(1)
   >> data(1).ppg
   >> data(1).ppg.fs
   >> size(data(1).ppg.v)

4. Set paths and run RRest:
   >> addpath('{RREST_PATH}')
   >> addpath('{RREST_PATH / 'Algorithms'}')
   >> cd '{output_mat_path.parent}'

5. Run RRest on this dataset:
   >> RRest('test_ppg_for_rrest')
   
   Note: RRest will look for 'test_ppg_for_rrest_data.mat'
   So rename the file to match, or adjust the name above.

6. Or manually test individual components:
   >> up = setup_universal_params('test_ppg_for_rrest');
   >> % Then inspect 'up' to see configuration
""")


EXPORTING PPG DATA FOR MATLAB/RREST

Using subject: id_0a73ef1b-da67-43ff-b61a-f98c151be799

Extracting 3 windows of 120s each...
  Window 1: 7680 samples (120.0s)
  Window 2: 7488 samples (117.0s)
  Window 3: 7168 samples (112.0s)

✓ Extracted 3 valid windows

✓ Saved MATLAB file: /Users/jithuazeez/Documents/Msc/Dissertation/reports/test_ppg_for_rrest.mat
  File size: 175.6 KB
  Number of recordings: 3

HOW TO USE IN MATLAB

1. Open MATLAB

2. Load the data:
   >> load('/Users/jithuazeez/Documents/Msc/Dissertation/reports/test_ppg_for_rrest.mat')
   >> whos data

3. Inspect the structure:
   >> data(1)
   >> data(1).ppg
   >> data(1).ppg.fs
   >> size(data(1).ppg.v)

4. Set paths and run RRest:
   >> addpath('/Users/jithuazeez/Documents/Msc/Dissertation/experiments/shared/RRest/RRest/RRest_v3.0')
   >> addpath('/Users/jithuazeez/Documents/Msc/Dissertation/experiments/shared/RRest/RRest/RRest_v3.0/Algorithms')
   >> cd '/Users/jithuazeez/Documents/Msc/Dissertation/reports'

5. Run RRes